# Computational Theory Assessment: Implementing SHA-256

In [ ]:
import hashlib
import math

import numpy as np

## Problem 1: Representing SHA-256 Data

Before implementing SHA-256, investigate how the input, output, and intermediate data can be represented in Python.

Explain how you would represent:

32-bit words,

sequences of 32-bit words,

input messages,

512-bit message blocks,

the final 256-bit hash value,

big-endian integers.

For each representation, explain why it is appropriate and demonstrate your choices with small Python examples.

### 32-bit words

A 32-bit word contains 32 bits or 4 bytes.

Bits: 1010 0001 0000 0011 1111 1110 0010 0011
Hexadecimal: a103fe23

According to the NIST Secure Hash Standard ([Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):
>"An integer between 0 and $2^{32} - 1$ inclusive may be represented as a 32-bit word.  The least significant four bits of the integer are represented by the right-most hex digit of the word representation. For example, the integer 291=28 + 25 + 21 + 20=256+32+2+1 is represented by the hex word “00000123” If Z is an integer, 0 ≤ Z < $2^{64}$, then Z=$2^{32}$X + Y, where 0 ≤ X < $2^{32}$ and 0 ≤ Y < $2^{32}$.  Since X and Y can be represented as 32-bit words x and y, respectively, the integer Z can be represented as the pair of words (x, y).  This property is used for SHA-1, SHA224 and SHA-256." 


In [ ]:
# create a variable for 32-bit word
# the smallest valid unsigned word in int
word = 0

print("Value:", word)

# show word in binary string
print("Binary:", bin(word)) 

# show size of word
print("Number of significant bits:", word.bit_length())
print("32-bit representation:", format(word, "032b"))

# the biggest valid unsigned word in int
word2 = 2**32 - 1

print("Value:", word2)

# show word in binary string
print("Binary:", bin(word2)) 

# show size of word
print("Number of significant bits:", word2.bit_length())
print("32-bit representation:", format(word2, "032b"))

# the first value out of range
word2 = 2**32 + 1

print("Value:", word2)

# show word in binary string
print("Binary:", bin(word2)) 

# show size of word
print("Number of significant bits:", word2.bit_length())
print("32-bit representation:", format(word2, "032b"))

Value: 0
Binary: 0b0
Number of significant bits: 0
32-bit representation: 00000000000000000000000000000000
Value: 4294967295
Binary: 0b11111111111111111111111111111111
Number of significant bits: 32
32-bit representation: 11111111111111111111111111111111
Value: 4294967297
Binary: 0b100000000000000000000000000000001
Number of significant bits: 33
32-bit representation: 100000000000000000000000000000001


#### Why did you choose Python's int?
I chose python's int as an int between 0 and $2^{32} - 1$ inclusive may be represented as a 32-bit word. However, Python integers have arbitrary precision, so using int alone does not restrict a value to 32 bits.

#### What is bit_length()?
bit_length() tells me the number of significant bits in the given data type, without the 0s. For example, the integer 5 has a bit length of 3 because its binary representation is 101.

#### Why format(word, "032b") is useful.
This auto formats the word into a minimum of 32 bits. It displays an integer in binary (b), with a minimum width of 32 characters (32) and leading zeros when needed (0).

#### Why formatting alone doesn't guarantee that a number fits within 32 bits.
Formatting alone does not guarantee a value fits within 32 bits because Python's format width is a minimum rather than a maximum, thus python can still display a value with 33 bits as it meets the minimum requirement. Therefore, we must also check that the integer lies between 0 and $2^{32} - 1$ inclusive.

### Sequences of 32 bit words

A sequence of 32 bit words is an ordered collection --> can be used for operations like addition, multiplication etc

Since 32-bit words can be represented as integers, a Python list of integers is a suitable representation for a sequence of words.

 *A list is suitable because it stores multiple values in order and allows me to iterate through each word easily. Python lists are flexible and do not require an additional library.*
 
However, each integer must be checked to ensure that it falls within the unsigned 32-bit range, from 0 to $(2^{32}-1)$

In [ ]:
# since one 32-bit word is best represented as an int
word1 = 5;
word2 = 10;
word3 = 15;

#create the collection of 32 bit words
sequence = [word1, word2, word3];

print(sequence);

# validate the sequence of 32-bit words
for word in sequence:
    if not 0 <= word <= 0xFFFFFFFF: #0xFFFFFFFF is a hex num where all 32 bits is set to 1
        raise ValueError("Value is outside the 32-bit unsigned range")

    print(format(word, "032b")) #print as a 32 bit binary word

[5, 10, 15]
00000000000000000000000000000101
00000000000000000000000000001010
00000000000000000000000000001111


In [ ]:
# considering out of range values 
word = 2**32;

try:
    if not 0 <= word <= 0xFFFFFFFF:
        raise ValueError("Value is outside the 32-bit unsigned range")

    print(format(word, "032b"))

except ValueError as error:
    print("Validation error:", error)

Validation error: Value is outside the 32-bit unsigned range


#### Why a for loop?
The for loop iterates through each element in the list and prints its 32-bit binary representation. Hence, when I add another word to the list, the loop processes it automatically, so I don't need to change the loop. This makes it easier to use.

### Input Messages

User prompt: A prompt such as "Enter a message:" asks the user to type something.

Input message: The actual text the user enters, such as "hello", which will be hashed.

**Does the message need a dynamic data type, since we do not know the length of the user's input in advance?**

*Python's bytes type can represent sequences of different lengths, so it is suitable for storing input messages of varying sizes.*

According to the NIST Secure Hash Standard ([Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):

>"For SHA-1, SHA-224 and SHA-256, each message block has 512 bits, which are represented as a sequence of sixteen 32-bit words."

This means that each message block contains 512 bits, arranged into 16 words of 32 bits each. 8 bits = 1 byte, hence each block conatins 64 bytes

*I think Python's bytes type is a suitable representation for input messages because it stores a sequence of byte values, each ranging from 0 to 255. If the input is text, it can first be converted into bytes using an encoding such as UTF-8.*

In [ ]:
#ordinary string
message = "happy days"
print(message)
print("Message type:", type(message))

#convert string to bytes
my_word = "hello"                 # str
my_bytes = my_word.encode("utf-8")  # bytes

# print the bytes representation of the string
print(my_bytes)
print("Message type:", type(my_bytes))
print("Message length:", len(my_bytes))

happy days
Message type: <class 'str'>
b'hello'
Message type: <class 'bytes'>
Message length: 5


This is useful because SHA-256 processes message data as bits, so text must be converted into bytes before it can be processed by the algorithm. Using Python's bytes type provides a suitable representation of the input message.

### 512-bit message blocks



The chunks of data by SHA-256 that are represented as a sequence of sixteen 32-bit words.

512 bits = 64 bytes per block.

According to the NIST Secure Hash Standard ([Section 5.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):

>"For SHA-1, SHA-224 and SHA-256, the message and its padding are parsed into N 512-bit blocks ... the 512 bits of the input block may be expressed as sixteen 32-bit word"

As established earlier for sequences of 32-bit words, this is best represented by a list. Therefore, a 512-bit message block could also be represented by a list containing 16 integers, with each integer representing one 32-bit word. 

*A list is convenient because it allows individual words to be accessed and used in the operations performed by SHA-256. The code below demonstrates this representation by creating a list of 16 integers and checking the number of words and total number of bits.*

In [ ]:
# since one 32-bit word is best represented as an int, create 16 words to represent a 512-bit message
sequence = []

# create a sequence of 16 32-bit words, by looping through the range of 1 to 16 and multiplying each value by 5
for i in range(1, 17):
    sequence.append(i * 5)

# print the sequence of 32-bit words
print(sequence)
print ("Number of words in sequence:", len(sequence))
print("Number of bits in sequence:", len(sequence) * 32)

[5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80]
Number of words in sequence: 16
Number of bits in sequence: 512


###  Final 256-bit hash value

According to the NIST Secure Hash Standard (Section 6.2):
>"The final result of SHA-256 is a 256-bit message digest"

This means that the final 256-bit hash value is the output SHA-256 produces after hashing the input message, by processing the 512-message blocks including the padding.

*Hence, Python's bytes type is a suitable representation because it stores a sequence of byte values, and each byte contains 8 bits.*

In [ ]:
#create a bytes object
message = b"hello"
digest = hashlib.sha256(message).digest()

# print the digest and its properties
print("Digest:", digest)
print("Digest type:", type(digest))
print("Digest length in bytes:", len(digest))
print("Digest length in bits:", len(digest) * 8)
print("Hexadecimal digest:", digest.hex())
print("Hexadecimal length:", len(digest.hex()))

Digest: b',\xf2M\xba_\xb0\xa3\x0e&\xe8;*\xc5\xb9\xe2\x9e\x1b\x16\x1e\\\x1f\xa7B^s\x043b\x93\x8b\x98$'
Digest type: <class 'bytes'>
Digest length in bytes: 32
Digest length in bits: 256
Hexadecimal digest: 2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
Hexadecimal length: 64


#### Why not an integer?
When you represent a hash as an integer, leading zeros can disappear as it does not preserved in its usual numerical representation, which can make its displayed bit length shorter than 256 bits. Representing the digest as bytes keeps its fixed length of 32 bytes (256 bits), including any leading zero bytes.

### Big-endian integers

According to the NIST Secure Hash Standard (Section 3.1):
>"Throughout this specification, the “big-endian” convention is used when expressing both 32- and 64-bit words, so that within each word, the most significant bit is stored in the left-most bit position."

Applying this concept by replacing bits with bytes, a Big-endian representation is a way of arranging bytes when representing an integer. The most significant byte (the byte with the greatest place value) comes first, followed by the remaining bytes in descending significance.

Python's int type can represent the numerical value, while bytes can represent its bytes in a specific order. The following experiment demonstrates how Python converts an integer into big-endian bytes and then converts it back into an integer.

In [ ]:
number = 0x12345678

# convert integer to bytes using big-endian byte order
data = number.to_bytes(4, byteorder="big")

# print the integer, its big-endian byte representation, and its hexadecimal representation
print("Integer:", number)
print("Big-endian bytes:", data)
print("Hexadecimal representation:", data.hex())
print("Back to integer:", int.from_bytes(data, "big"))

Integer: 305419896
Big-endian bytes: b'\x124Vx'
Hexadecimal representation: 12345678
Back to integer: 305419896


## Problem 2: SHA-256 Bitwise Operations

Implement the Boolean and bitwise functions used by SHA-256:

1. `Ch(x, y, z)`
2. `Maj(x, y, z)`
3. `Sigma0(x)` — written as $\Sigma_0^{256}(x)$ in the standard
4. `Sigma1(x)` — written as $\Sigma_1^{256}(x)$ in the standard
5. `sigma0(x)` — written as $\sigma_0^{256}(x)$ in the standard
6. `sigma1(x)` — written as $\sigma_1^{256}(x)$ in the standard

Use NumPy so that the values involved in these operations are represented as 32-bit unsigned integers.

For each function:
- provide an appropriate docstring,
- explain the operation in Markdown,
- implement the function directly from that definition,
- provide tests on appropriate inputs.

According to the NIST Secure Hash Standard ([Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):
>"SHA-224 and SHA-256 both use six logical functions, where each function operates on 32-bit words, which are represented as x, y, and z. The result of each function is a new 32-bit word."

**How do I represent 32-bit unsigned integers?**

They should be represented as np.uint32(num). ([Source Documentation](https://numpy.org/devdocs/user/basics.types.html))

**Relevant Bitwise Truth Tables**
    
<img src="images/TruthTables.jpeg" width="500">

[Representations of Logic Operations, Truth tables and their relation to logic circuits and equations](https://www.realdigital.org/doc/e127ebfa82dbc904b5c0dac5d1adce8e)

### Ch(x, y, z)
**What is a Ch() function?**

It is a choose (ch) function in SHA-256 where it uses x to choose between y and z, bit by bit.

According to the NIST Secure Hash Standard ([Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):
$$
\operatorname{Ch}(x,y,z) = (x \land y) \oplus (\neg x \land z)
$$

**What do the symbols mean?**

According to the NIST Secure Hash Standard ([Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):
- $\land$ — Bitwise AND operation.
- $\oplus$ — Bitwise XOR (“exclusive-OR”) operation.
- $\neg$ — Bitwise complement operation.

**How do I apply these symbols to the given formula, step by step?**

- x $\land$ y -> x AND y 
- $\neg$ x $\land$ z -> complement x AND z
- $(x \land y) \oplus (\neg x \land z)$ -> (x AND y) XOR (omplement x AND z)

With reference to the truth tables, for $\operatorname{Ch}(1,1,0)$:

$$
\begin{aligned}
\operatorname{Ch}(1,1,0)
&= (1 \land 1) \oplus (\neg 1 \land 0) \\
&= (1 \land 1) \oplus (0 \land 0) \\
&= 1 \oplus 0 \\
&= 1
\end{aligned}
$$


In [ ]:
def Ch(x, y, z):
    """Uses each bit of x to choose the corresponding bit from y or z."""
    return (x & y) ^ (~x & z)

In [ ]:
# checking that ch() chooses bits from y when x is 1
# representing as 32-bit unsigned integers
x = np.uint32(0xFFFF0000)
y = np.uint32(0x12345678)
z = np.uint32(0x9ABCDEF0)

# Check that the result is as expected
assert Ch(x, y, z) == np.uint32(0x1234DEF0)

# Should return 1 since x is 1 and y is 1, so the result should be 1.
print("Result from Ch() function: " + str(Ch(x, y, z)))

Result from ch() function: 305454832


### Maj(x, y, z)
**What is a Maj() function?**

It is a majority (maj) function, it takes three 32-bit words, x, y, and z, and examines their bits position by position. It returns the bit that appears at least twice among x, y and z.

According to the NIST Secure Hash Standard ([Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):
$$
\operatorname{Maj}(x,y,z) = (x \land y) \oplus (x \land z) \oplus (y \land z)
$$

**What do the symbols mean?**

According to the NIST Secure Hash Standard ([Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):
- $\land$ — Bitwise AND operation.
- $\oplus$ — Bitwise XOR (“exclusive-OR”) operation.

**How do I apply these symbols to the given formula, step by step?**

- x $\land$ y -> x AND y 
- $(x \land y) \oplus (\neg x \land z)$ -> (x AND y) XOR (x AND z) XOR (y AND z)

With reference to the truth tables, for $\operatorname{Maj}(1,1,0)$:

$$
\begin{aligned}
\operatorname{Maj}(1,1,0)
&= (1 \land 1) \oplus (1 \land 0) \oplus (1 \land 0) \\
&= 1 \oplus 0 \oplus 0\\
&= 1
\end{aligned}
$$

In [ ]:
def Maj(x, y, z):
    """Returns the majority of the three inputs, bit by bit."""
    return (x & y) ^ (x & z) ^ (y & z)

In [ ]:
# representing as 32-bit unsigned integers
x = np.uint32(0xFFFF0000)
y = np.uint32(0x12345678)
z = np.uint32(0x9ABCDEF0)

# Check that the result is as expected
assert Maj(x, y, z) == np.uint32(0x9ABC5670)

# Should return 0 since the majority of the bits are 0
print("Result from Maj() function: " + str(Maj(x, y, z)))

Result from maj() function: 2596034160


#### Relevant Bitwise Functions
**What does ROTR mean?** 

According to the NIST Secure Hash Standard ([Section 3.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):

> The rotate right (circular right shift) operation, where \(x\) is a \(w\)-bit word and \(n\) is an integer with \(0 \leq n < w\), is defined by $$\operatorname{ROTR}^{n}(x) = (x \gg n) \lor (x \ll (w-n)).$$ Thus, $\operatorname{ROTR}^{n}(x)$ is equivalent to a circular shift *(rotation) of x by n positions* to the right.

The right-most bit then shifts to the left-most bit position.

In [ ]:
def rotr(x, n):
    """Performs a right rotation on a 32-bit word."""
    x = np.uint32(x)
    return (x >> n) | (x << (32 - n))

**What is SHR?**

According to the NIST Secure Hash Standard ([Section 3.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):

> The right shift operation $\operatorname{SHR}^{n}(x)$, where \(x\) is a \(w\)-bit word and \(n\) is an integer with \(0 \leq n < w\), is defined by $$\operatorname{SHR}^{n}(x) = x \gg n.$$ This operation is used in the SHA-224, SHA-256, SHA-384, SHA-512, SHA-512/224 and SHA-512/256 algorithms.

Hence its like this symbol, >>

In [ ]:
# taken from notes/sha512.ipynb
def shr(x, n):
    """Logical right shift of a 32-bit unsigned word."""
    return np.uint32(x) >> n

In [ ]:
# tests for rotr and shr functions
assert rotr(np.uint32(1), 1) == np.uint32(0x80000000)   # low bit wraps to the top
assert shr(np.uint32(0x80000001), 3) == np.uint32(0x10000000)  # low bits are dropped

### Sigma0(x) — written as $\Sigma_0^{256}(x)$ in the standard
**What is a Sigma0(x) function?**

According to the NIST Secure Hash Standard ([Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):

$$
\Sigma_0^{256}(x) =
\operatorname{ROTR}^{2}(x)
\oplus \operatorname{ROTR}^{13}(x)
\oplus \operatorname{ROTR}^{22}(x)
$$

**How does ROTR relate to our current Sigma0(x)?**

If n is the number of positions x will be moved to the right,

- $\operatorname{ROTR}^{2}(x)$ -> circular right shift of x by 2 positions
- $\operatorname{ROTR}^{13}(x)$ -> circular right shift of x by 13 positions
- $\operatorname{ROTR}^{22}(x)$ -> circular right shift of x by 22 positions


In [ ]:
# representing as 32-bit unsigned integers
x = np.uint32(0x80000001)

def Sigma0(x):
    """Computes the Sigma0 function as defined in SHA-256."""
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

print("Result from Sigma0() function: " + str(Sigma0(x)))

Result from Sigma0() function: 1611400704


### Sigma1(x) — written as $\Sigma_1^{256}(x)$ in the standard

**What is a Sigma1(x) function?**

According to the NIST Secure Hash Standard ([Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):

$$
\Sigma_1^{256}(x) =
\operatorname{ROTR}^{6}(x)
\oplus \operatorname{ROTR}^{11}(x)
\oplus \operatorname{ROTR}^{25}(x)
$$

**How does ROTR relate to our current Sigma0(x)?**

If n is the number of positions x will be moved to the right,

- $\operatorname{ROTR}^{6}(x)$ -> circular right shift of x by 6 positions
- $\operatorname{ROTR}^{11}(x)$ -> circular right shift of x by 11 positions
- $\operatorname{ROTR}^{25}(x)$ -> circular right shift of x by 25 positions

In [ ]:
# representing as 32-bit unsigned integers
x = np.uint32(0x80000001)

def Sigma1(x):
    """Computes the Sigma1 function as defined in SHA-256."""
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

print("Result from Sigma1() function: " + str(Sigma1(x)))

Result from Sigma1() function: 103809216


### sigma0(x) — written as $\sigma_0^{256}(x)$ in the standard
**What is a sigma0(x) function?**

According to the NIST Secure Hash Standard ([Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):

$$
\sigma_0^{256}(x) =
\operatorname{ROTR}^{7}(x)
\oplus \operatorname{ROTR}^{18}(x)
\oplus \operatorname{SHR}^{3}(x)
$$

**How does SHR currently relate to our sigma0(x)?**
- $\operatorname{ROTR}^{7}(x)$ -> circular right shift of x by 7 positions
- $\operatorname{ROTR}^{18}(x)$ -> circular right shift of x by 18 positions
- $\operatorname{SHR}^{3}(x)$ -> discards 3 right most bits, inserts zeroes on the left

In [ ]:
# representing as 32-bit unsigned integers
x = np.uint32(0x80000001)

def sigma0(x):
    """Computes the sigma0 function as defined in SHA-256."""
    return rotr(x, 7) ^ rotr(x, 18) ^ shr(x, 3)

print("Result from sigma0() function: " + str(sigma0(x)))

Result from sigma0() function: 318791680


### sigma1(x) — written as $\sigma_1^{256}(x)$ in the standard
**What is a sigma1(x) function?**

According to the NIST Secure Hash Standard ([Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):

$$
\sigma_1^{256}(x) =
\operatorname{ROTR}^{17}(x)
\oplus \operatorname{ROTR}^{19}(x)
\oplus \operatorname{SHR}^{10}(x)
$$

**How does SHR currently relate to our sigma0(x)?**
- $\operatorname{ROTR}^{17}(x)$ -> circular right shift of x by 17 positions
- $\operatorname{ROTR}^{19}(x)$ -> circular right shift of x by 19 positions
- $\operatorname{SHR}^{10}(x)$ -> discards 10 right most bits, inserts zeroes on the left


In [ ]:
# representing as 32-bit unsigned integers
x = np.uint32(0x80000001)

def sigma1(x):
    """Computes the sigma1 function as defined in SHA-256."""
    return rotr(x, 17) ^ rotr(x, 19) ^ shr(x, 10)

print("Result from sigma1() function: " + str(sigma1(x)))

Result from sigma1() function: 2158592


## Problem 3: Generating the SHA-256 Constants
The Secure Hash Standard defines the initial hash values and round constants used by SHA-256 using fractional parts of square and cube roots of prime numbers.

Use Python and NumPy to generate the 64 round constants specified in the standard.

Your solution should:

1. write a function primes(n) that generates the first n prime numbers,
2. calculate the cube roots of the first 64 primes,
3. extract the fractional part of each cube root,
4. convert the resulting to a 32-bit unsigned integer,
5. display the resulting constants in hexadecimal,
6. compare your results with the constants in the Secure Hash Standard.

### Write a function primes(n) that generates the first n prime numbers

In [ ]:
# Brute force test for primality.
def isprime(n):
    """Test whether n is prime."""
    # Loop through 2...n-1.
    for i in range(2, n):
        # Calculate remainder of n divided by i.
        if n % i == 0:
            # If this is zero, then n is not prime.
            return False
    # If we get here, then n is prime.
    return True

In [ ]:
def primes(n):
    """generates the first n prime numbers"""
    primes_arr = []
    num = 2  # Start checking for primes from the first prime number
    while len(primes_arr) < n:
            if isprime(num):
                primes_arr.append(num)
            num += 1
    return primes_arr

primes(64)
p = primes(64)

print("The first 64 prime numbers are: " + str(p))

The first 64 prime numbers are: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311]


### Calculate the cube roots of the first 64 primes

#### How do we get a cube root in python?
You can use math.pow(base, exponent) -> [GeeksForGeeks](https://www.geeksforgeeks.org/python/python-program-for-find-cubic-root-of-a-number/)


In [ ]:
def cube_root(n):
    """generates the cube roots of a given set of nums"""
    cube_roots = []
    for i in n:
        a = math.pow(i, (1/3)) 
        cube_roots.append(a)
    return cube_roots

#prints out first 64 prime numbers as cube roots
roots = cube_root(p)
print("The cube roots of the first 64 prime numbers are: " + str(roots))

The cube roots of the first 64 prime numbers are: [1.2599210498948732, 1.4422495703074083, 1.7099759466766968, 1.912931182772389, 2.2239800905693152, 2.3513346877207573, 2.571281590658235, 2.668401648721945, 2.8438669798515654, 3.072316825685847, 3.1413806523913927, 3.332221851645953, 3.4482172403827303, 3.503398060386724, 3.6088260801386944, 3.756285754221072, 3.8929964158732604, 3.936497183102173, 4.0615481004456795, 4.140817749422853, 4.179339196381232, 4.290840427026207, 4.362070671454838, 4.464745095584537, 4.594700892207039, 4.657009507803835, 4.687548147653597, 4.7474593985234, 4.776856181035017, 4.834588127111639, 5.026525695313479, 5.0787530781327, 5.155136735475772, 5.180101467380292, 5.301459192380904, 5.325074021614986, 5.394690712109591, 5.462555571281397, 5.506878446387352, 5.5720546555426225, 5.635740794544236, 5.65665282582291, 5.758965220492401, 5.778996565152129, 5.818647867496961, 5.838272460814002, 5.953341813139051, 6.064126994506963, 6.100170200393062, 6.118033172

### Extract the fractional part of each cube root

#### What does fractional part mean?
It refers to the decimal or non-integer "leftover" part that remains after you subtract the whole number (integer) part of that cube root. ([Documentation](https://brilliant.org/wiki/factional-part-function/))

In [ ]:
def extract_fractional_part(n):
    """extracts the fractional part of a given set of nums"""
    float_arr = []
    for i in n:
        a = i - int(i)
        float_arr.append(a)
    return float_arr

#prints out the extracted fractional parts 
fractions = extract_fractional_part(roots)
print("The fractional parts of the cube roots are: " + str(fractions))

The fractional parts of the cube roots are: [0.2599210498948732, 0.4422495703074083, 0.7099759466766968, 0.9129311827723889, 0.22398009056931523, 0.35133468772075727, 0.5712815906582351, 0.6684016487219449, 0.8438669798515654, 0.07231682568584707, 0.14138065239139275, 0.33222185164595297, 0.44821724038273025, 0.5033980603867239, 0.6088260801386944, 0.7562857542210719, 0.8929964158732604, 0.9364971831021731, 0.06154810044567949, 0.14081774942285286, 0.17933919638123186, 0.2908404270262066, 0.3620706714548376, 0.46474509558453736, 0.5947008922070394, 0.6570095078038349, 0.6875481476535974, 0.7474593985234002, 0.7768561810350167, 0.8345881271116387, 0.026525695313479147, 0.07875307813270016, 0.15513673547577156, 0.1801014673802923, 0.3014591923809036, 0.3250740216149861, 0.3946907121095906, 0.46255557128139735, 0.5068784463873524, 0.5720546555426225, 0.6357407945442359, 0.6566528258229098, 0.7589652204924011, 0.7789965651521289, 0.8186478674969608, 0.838272460814002, 0.9533418131390512, 0

### Convert the resulting to a 32-bit unsigned integer

We need to scale the fractional part values into integers by \(2^{32}\).

#### Why multiply by \(2^{32}\)?

A 32-bit unsigned integer can represent values from:

$$
0 \text{ to } 2^{32}-1
$$

Multiplying the fractional part by \(2^{32}\) scales it into the 32-bit integer range.

For example:

$$
0.259921 \times 2^{32} \approx 1116352408
$$


In [ ]:
# 2 lines below this are taken from chatgpt to fix my bug where the fractional parts were not being scaled to 32-bit unsigned integers
scaled = [i * (2**32) for i in fractions] 
integer_values = [int(i) for i in scaled]

def convert_to_32_bit(n):
    """generate the 32-bit unsigned integer representation of the given set of nums"""
    int_arr = []
    for i in n:
        if not 0 <= i <= 0xFFFFFFFF: #0xFFFFFFFF is a hex num where all 32 bits is set to 1
            raise ValueError("Value is outside the 32-bit unsigned range")
        else:
            a = np.uint32(i)
            int_arr.append(a)
    return int_arr

print("The 32-bit unsigned integer representations are: " + str(convert_to_32_bit(integer_values)))
bits_32 = convert_to_32_bit(integer_values)


The 32-bit unsigned integer representations are: [np.uint32(1116352408), np.uint32(1899447441), np.uint32(3049323471), np.uint32(3921009573), np.uint32(961987163), np.uint32(1508970993), np.uint32(2453635748), np.uint32(2870763221), np.uint32(3624381080), np.uint32(310598401), np.uint32(607225278), np.uint32(1426881987), np.uint32(1925078388), np.uint32(2162078206), np.uint32(2614888103), np.uint32(3248222580), np.uint32(3835390401), np.uint32(4022224774), np.uint32(264347078), np.uint32(604807628), np.uint32(770255983), np.uint32(1249150122), np.uint32(1555081692), np.uint32(1996064986), np.uint32(2554220882), np.uint32(2821834349), np.uint32(2952996808), np.uint32(3210313671), np.uint32(3336571891), np.uint32(3584528711), np.uint32(113926993), np.uint32(338241895), np.uint32(666307205), np.uint32(773529912), np.uint32(1294757372), np.uint32(1396182291), np.uint32(1695183700), np.uint32(1986661051), np.uint32(2177026350), np.uint32(2456956037), np.uint32(2730485921), np.uint32(2820302

In [ ]:
# Testing
numbers = [1, 100, 500]
converted = convert_to_32_bit(numbers)

print(converted)
print(type(converted[0]))

[np.uint32(1), np.uint32(100), np.uint32(500)]
<class 'numpy.uint32'>


### Display the resulting constants in hexadecimal

#### What is a hexadecimal number?
Hexadecimal is a positional numeral system for representing a numeric value as base 16 ([Source](https://en.wikipedia.org/wiki/Hexadecimal))

In [ ]:
def hexadecimal(n):
    """generate the hexadecimal representation of the given set of nums"""
    int_arr = []
    for i in n:
        if not isinstance(i, (int, np.integer)):
            raise ValueError("Value is not an integer")
        if not 0 <= i <= 0xFFFFFFFF: #0xFFFFFFFF is a hex num where all 32 bits is set to 1
            raise ValueError("Value is outside the 32-bit unsigned range")
        else:
            a = format(i, "08x") #format as 8-digit hexadecimal
            int_arr.append(a)
    return int_arr

print("The hexadecimal representations are: " + str(hexadecimal(bits_32)))
result = hexadecimal(bits_32)

The hexadecimal representations are: ['428a2f98', '71374491', 'b5c0fbcf', 'e9b5dba5', '3956c25b', '59f111f1', '923f82a4', 'ab1c5ed5', 'd807aa98', '12835b01', '243185be', '550c7dc3', '72be5d74', '80deb1fe', '9bdc06a7', 'c19bf174', 'e49b69c1', 'efbe4786', '0fc19dc6', '240ca1cc', '2de92c6f', '4a7484aa', '5cb0a9dc', '76f988da', '983e5152', 'a831c66d', 'b00327c8', 'bf597fc7', 'c6e00bf3', 'd5a79147', '06ca6351', '14292967', '27b70a85', '2e1b2138', '4d2c6dfc', '53380d13', '650a7354', '766a0abb', '81c2c92e', '92722c85', 'a2bfe8a1', 'a81a664b', 'c24b8b70', 'c76c51a3', 'd192e819', 'd6990624', 'f40e3585', '106aa070', '19a4c116', '1e376c08', '2748774c', '34b0bcb5', '391c0cb3', '4ed8aa4a', '5b9cca4f', '682e6ff3', '748f82ee', '78a5636f', '84c87814', '8cc70208', '90befffa', 'a4506ceb', 'bef9a3f7', 'c67178f2']


### Compare your results with the constants in the Secure Hash Standard

Grabbing the constants from the NIST Extract [Section 4.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)

#### 4.2.2 SHA-224 and SHA-256 Constants

SHA-224 and SHA-256 use the same sequence of sixty-four constant 32-bit words:

$$
K_0^{(256)}, K_1^{(256)}, \ldots, K_{63}^{(256)}
$$

These words represent the first 32 bits of the fractional parts of the cube roots of the first 64 prime numbers.

The constants are represented in hexadecimal. Listed below are the 64 constant words, in order:

```text
428a2f98 71374491 b5c0fbcf e9b5dba5
3956c25b 59f111f1 923f82a4 ab1c5ed5
d807aa98 12835b01 243185be 550c7dc3
72be5d74 80deb1fe 9bdc06a7 c19bf174
e49b69c1 efbe4786 0fc19dc6 240ca1cc
2de92c6f 4a7484aa 5cb0a9dc 76f988da
983e5152 a831c66d b00327c8 bf597fc7
c6e00bf3 d5a79147 06ca6351 14292967
27b70a85 2e1b2138 4d2c6dfc 53380d13
650a7354 766a0abb 81c2c92e 92722c85
a2bfe8a1 a81a664b c24b8b70 c76c51a3
d192e819 d6990624 f40e3585 106aa070
19a4c116 1e376c08 2748774c 34b0bcb5
391c0cb3 4ed8aa4a 5b9cca4f 682e6ff3
748f82ee 78a5636f 84c87814 8cc70208
90befffa a4506ceb bef9a3f7 c67178f2

In [ ]:
official_constants = """
428a2f98 71374491 b5c0fbcf e9b5dba5
3956c25b 59f111f1 923f82a4 ab1c5ed5
d807aa98 12835b01 243185be 550c7dc3
72be5d74 80deb1fe 9bdc06a7 c19bf174
e49b69c1 efbe4786 0fc19dc6 240ca1cc
2de92c6f 4a7484aa 5cb0a9dc 76f988da
983e5152 a831c66d b00327c8 bf597fc7
c6e00bf3 d5a79147 06ca6351 14292967
27b70a85 2e1b2138 4d2c6dfc 53380d13
650a7354 766a0abb 81c2c92e 92722c85
a2bfe8a1 a81a664b c24b8b70 c76c51a3
d192e819 d6990624 f40e3585 106aa070
19a4c116 1e376c08 2748774c 34b0bcb5
391c0cb3 4ed8aa4a 5b9cca4f 682e6ff3
748f82ee 78a5636f 84c87814 8cc70208
90befffa a4506ceb bef9a3f7 c67178f2
"""

official_constants = official_constants.split()

In [ ]:
#Compare both results 

def compare(arr1, arr2):
    if np.array_equal(arr1, arr2):
        return True
    else:
        return False

print("Comparing the generated constants with the official constants: " + str(compare(result, official_constants)))

Comparing the generated constants with the official constants: True


## Problem 4: Padding and Parsing Messages

Write a generator function called block_parse(msg) that accepts a bytes object and yields the 512-bit message blocks required by SHA-256.

Each yielded block should contain sixteen 32-bit unsigned integers.

Implement the padding specified in sections 5.1.1 and 5.2.1 of the Secure Hash Standard, including:

- appending the 1 bit,
- appending the required zero bits,
- appending the original message length as a 64-bit integer,
- handling messages that require one or two final blocks,
- interpreting the message using the required byte order.

Test your implementation, including any boundary cases you can identify.